In [2]:
from PIL import Image
import depth_pro

# Load model and preprocessing transform
model, transform = depth_pro.create_model_and_transforms()
model.to('cuda')



DepthPro(
  (encoder): DepthProEncoder(
    (patch_encoder): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linea

In [3]:
path = "./data/test1.jpeg"

In [4]:
import torch
print(torch.cuda.is_available())       # True if CUDA is available
print(torch.cuda.current_device())     # Device index
print(torch.cuda.device_count())       # Number of GPUs


True
0
1


In [5]:
image, _, f_px = depth_pro.load_rgb(path)
image = transform(image)
image = image.to("cuda")
prediction = model.infer(image, f_px=f_px)
depth = prediction["depth"]  # Depth in [m]

print(depth)

tensor([[0.8326, 0.8311, 0.8229,  ..., 2.8687, 2.7308, 2.7064],
        [0.8271, 0.8256, 0.8176,  ..., 2.9305, 2.8355, 2.8184],
        [0.8166, 0.8151, 0.8075,  ..., 3.0584, 3.0633, 3.0643],
        ...,
        [0.7555, 0.7552, 0.7532,  ..., 2.4462, 2.4372, 2.4355],
        [0.7578, 0.7577, 0.7575,  ..., 2.3284, 2.2234, 2.2047],
        [0.7589, 0.7590, 0.7597,  ..., 2.2720, 2.1272, 2.1021]],
       device='cuda:0')


In [6]:
from ultralytics import YOLO
import cv2

obj_model = YOLO("yolov8n.pt")

In [7]:
results = obj_model(path, show=True, save=True)


image 1/1 c:\Users\joshu\Workspace\Depth Model\data\test1.jpeg: 480x640 1 bench, 4 chairs, 5 dining tables, 117.5ms
Speed: 9.3ms preprocess, 117.5ms inference, 99.8ms postprocess per image at shape (1, 3, 480, 640)
Results saved to runs\detect\predict11


In [8]:
from PIL import Image

img = Image.fromarray(results[0].plot())
img_resized = img.resize((img.width // 2, img.height // 2))  # half size
img_resized.show()


In [9]:


boxes = results[0].boxes  # boxes object
classes = results[0].boxes.cls.cpu().numpy()  # class IDs as numpy array


In [10]:
# COCO class names for YOLOv5/YOLOv8
class_names = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 
               'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 
               'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 
               'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 
               'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 
               'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 
               'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 
               'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

for box, cls in zip(boxes, classes):
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
    cx = int((x1 + x2) / 2)
    cy = int((y1 + y2) / 2)
    
    class_name = class_names[int(cls)]  # map class ID to name
    
    print(f"Detected object: {class_name}")
    print(f"Center of detected object: ({cx}, {cy})")
    print(f"Depth at center: {depth[cy, cx].item()} m")

# Code for using minimum in bounding box (doesn't work in cases where box is much larger than object)
# for box, cls in zip(boxes, classes):
#     # Extract bounding box coordinates
#     x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
    
#     # Convert to integer pixel indices, clamp to depth map size
#     x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
#     x1 = max(x1, 0)
#     y1 = max(y1, 0)
#     x2 = min(x2, depth.shape[1] - 1)  # width
#     y2 = min(y2, depth.shape[0] - 1)  # height

#     # Slice the depth map for the region inside the bounding box
#     box_depth_region = depth[y1:y2+1, x1:x2+1]  # rows, cols

#     # Get the minimum depth in the box
#     min_depth = box_depth_region.min().item()

#     # Map class ID to name
#     class_name = class_names[int(cls)]

#     print(f"Detected object: {class_name}")
#     print(f"Bounding box: ({x1}, {y1}) to ({x2}, {y2})")
#     print(f"Minimum depth in bounding box: {min_depth:.3f} m")


Detected object: chair
Center of detected object: (1662, 1856)
Depth at center: 3.477924108505249 m
Detected object: chair
Center of detected object: (1029, 1404)
Depth at center: 6.428588390350342 m
Detected object: chair
Center of detected object: (2549, 1720)
Depth at center: 4.224034786224365 m
Detected object: dining table
Center of detected object: (2048, 1752)
Depth at center: 5.546656608581543 m
Detected object: dining table
Center of detected object: (1511, 1484)
Depth at center: 5.640472412109375 m
Detected object: dining table
Center of detected object: (1481, 1376)
Depth at center: 4.988417148590088 m
Detected object: bench
Center of detected object: (3175, 2457)
Depth at center: 1.8612772226333618 m
Detected object: chair
Center of detected object: (861, 1314)
Depth at center: 8.316774368286133 m
Detected object: dining table
Center of detected object: (1484, 1299)
Depth at center: 5.457120418548584 m
Detected object: dining table
Center of detected object: (2049, 1493)
De

In [12]:
def generate_llm_prompt(boxes, classes, depth, class_names):
    detected_objects_info = []

    for box, cls in zip(boxes, classes):
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
        
        class_name = class_names[int(cls)]  # map class ID to name
        
        # Ensure center coordinates are within depth map bounds
        cx = max(0, min(cx, depth.shape[1] - 1))
        cy = max(0, min(cy, depth.shape[0] - 1))
        
        depth_at_center = depth[cy, cx].item()

        detected_objects_info.append(f"- A {class_name} located at approximate coordinates ({cx}, {cy}) with an estimated depth of {depth_at_center:.2f} meters.")

    if detected_objects_info:
        prompt = "Based on the visual analysis of the environment, the following objects have been detected:\n"
        prompt += "\n".join(detected_objects_info)
        prompt += "\nThis information provides a spatial understanding of the scene. The image shows the user facing directly ahead. Without ever referencing exact coordinates (depth in meters is fine) you will now utilize this information to help guide a blind user. Talk naturally, like a real-time human assistant.\n Respond with 1-2 sentences."
    else:
        prompt = "No distinct objects were detected in the environment, indicating a potentially clear or unpopulated scene."
    
    return prompt

# Example usage:
llm_prompt = generate_llm_prompt(boxes, classes, depth, class_names)
print(llm_prompt)

Based on the visual analysis of the environment, the following objects have been detected:
- A chair located at approximate coordinates (1662, 1856) with an estimated depth of 3.48 meters.
- A chair located at approximate coordinates (1029, 1404) with an estimated depth of 6.43 meters.
- A chair located at approximate coordinates (2549, 1720) with an estimated depth of 4.22 meters.
- A dining table located at approximate coordinates (2048, 1752) with an estimated depth of 5.55 meters.
- A dining table located at approximate coordinates (1511, 1484) with an estimated depth of 5.64 meters.
- A dining table located at approximate coordinates (1481, 1376) with an estimated depth of 4.99 meters.
- A bench located at approximate coordinates (3175, 2457) with an estimated depth of 1.86 meters.
- A chair located at approximate coordinates (861, 1314) with an estimated depth of 8.32 meters.
- A dining table located at approximate coordinates (1484, 1299) with an estimated depth of 5.46 meters.
